1. Création des nouveaux features

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = pd.read_csv("../data/df_clean.csv")

#Feature 1 : ratio impressions hors horaires : imprimer la nuit = tentative d'exfiltration
df["off_hours_print_ratio"] = (df["num_printed_pages_off_hours"] / (df["total_printed_pages"] + 1))

#Feature 2 : intensité de destruction de fichiers : brûler bcp de fichiers en peu de visites
df["burn_intensity"] = (df["total_files_burned"] / (df["num_entries"] + 1))

#Feature 3 : voyageur à haut risque : présence physique dans un pays hostile
df["is_high_risk_traveler"]=((df["is_abroad"]==1) & (df["hostility_country_level"]>= 2)).astype(int)

#Feature 4 : score d'activité physique anormale : combinaison de signaux faibles en un indicateur global
df["activity_score"] = (
        df["num_entries"]          * 2 +
        df["num_unique_campus"]    * 3 +
        df["entry_during_weekend"] * 5 +
        df["late_exit_flag"]       * 4    )

#Feature 5 : score d'exfiltration de données : poids basés sur les corrélations EDA avec is_malicious
df["data_exfiltration_score"] = (
        df["total_files_burned"]          * 3 +
        df["burned_from_other"]           * 5 +
        df["total_printed_pages"]         * 1 +
        df["num_printed_pages_off_hours"] * 4   )

print(f"Nombre de colonnes après création des features : {len(df.columns)}")

Nombre de colonnes après création des features : 27


2. Sélection des features

In [2]:

#Corrélation de chaque feature avec la cible 
corr_target = df.select_dtypes("number").corr()["is_malicious"].drop("is_malicious")
corr_sorted = corr_target.sort_values(ascending=False)

print("Corrélation  avec is_malicious :")
print(corr_sorted)


Corrélation  avec is_malicious :
data_exfiltration_score        0.466698
total_files_burned             0.407221
burn_intensity                 0.332690
burned_from_other              0.297451
num_printed_pages_off_hours    0.142346
activity_score                 0.138113
num_unique_campus              0.122642
total_printed_pages            0.116815
hostility_country_level        0.109457
entry_during_weekend           0.105110
is_high_risk_traveler          0.104830
num_entries                    0.101549
off_hours_print_ratio          0.072514
employee_classification        0.049519
employee_seniority_years       0.045448
has_foreign_citizenship        0.020623
is_abroad                      0.016589
trip_day_number                0.014995
has_medical_history            0.008291
is_contractor                 -0.029547
has_criminal_record           -0.034660
late_exit_flag                      NaN
Name: is_malicious, dtype: float64


In [3]:

#Calcul des corrélations entre elles
corr_matrix = df.select_dtypes("number").drop(columns=["is_malicious"]).corr()

# Identification des features quasi-inutiles
features_inutiles = corr_sorted[(corr_sorted.abs() < 0.03) | (corr_sorted.isna())].index.tolist()
#Identification des paires trop corrélées 
seuil_redondance = 0.90
paires_redondantes = []

print("Analyse des redondances :")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > seuil_redondance:
            col_i = corr_matrix.columns[i]
            col_j = corr_matrix.columns[j]
            
            if abs(corr_target[col_i]) < abs(corr_target[col_j]):
                a_supprimer = col_i
            else:
                a_supprimer = col_j
                
            paires_redondantes.append((col_i, col_j, a_supprimer))
            print(f"Redondance trouvée : {col_i} & {col_j}  =>  supprimer : {a_supprimer}")



Analyse des redondances :
Redondance trouvée : total_files_burned & burn_intensity  =>  supprimer : burn_intensity
Redondance trouvée : total_files_burned & data_exfiltration_score  =>  supprimer : total_files_burned
Redondance trouvée : hostility_country_level & is_high_risk_traveler  =>  supprimer : is_high_risk_traveler
Redondance trouvée : num_unique_campus & activity_score  =>  supprimer : num_unique_campus


In [4]:
# Extraction des features redondantes à supprimer
features_redondantes = list(set([p[2] for p in paires_redondantes]))

print(f"Features inutiles : {features_inutiles}")
print()
print(f"Features redondantes : {features_redondantes}")
print()

# Combinaison et suppression
cols_a_supprimer = list(set(features_inutiles + features_redondantes))

df = df.drop(columns=cols_a_supprimer)
print(f"{len(cols_a_supprimer)} features supprimées")
print()
print(f"Nombre de features prêtes pour le modèle : {len(df.columns)}") 

Features inutiles : ['has_foreign_citizenship', 'is_abroad', 'trip_day_number', 'has_medical_history', 'is_contractor', 'late_exit_flag']

Features redondantes : ['burn_intensity', 'total_files_burned', 'is_high_risk_traveler', 'num_unique_campus']

10 features supprimées

Nombre de features prêtes pour le modèle : 17


In [5]:
df.to_csv("../data/df_features.csv", index=False)